# Speaker Bio Card Generator
Generate character bio cards for each unique speaker from training data. These cards will be used by the council to inform emotion classification decisions.

In [ ]:
import sys
import os
from groq import Groq
from dotenv import load_dotenv
import pandas as pd
import json

# Setup paths and load environment
load_dotenv()
sys.path.append('../')
from src.load_data import load_data_from_csv

# Load API keys
BIO_CARD_API = os.getenv("BIO_CARD_GENERATOR")

print(f"✓ API key loaded: {BIO_CARD_API[:20]}..." if BIO_CARD_API else "✗ API key not found")

In [ ]:
# Load training data
training_data = load_data_from_csv('../data/train_sent_emo.csv')
training_data.drop_duplicates(inplace=True)

print(f"Loaded {len(training_data)} training samples")
print(f"\nUnique speakers: {training_data['Speaker'].nunique()}")
print(f"Speakers: {sorted(training_data['Speaker'].unique())}")
print(f"\nData shape: {training_data.shape}")
print(f"\nColumns: {training_data.columns.tolist()}")

In [ ]:
def groq_llm_call(prompt, model, api_key):
    """Call Groq API with retry logic."""
    client = Groq(api_key=api_key)
    chat_completion = client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
        model=model,
        temperature=1,
        max_tokens=2048
    )
    return chat_completion.choices[0].message.content

# Test the function
print("✓ groq_llm_call function defined")

In [ ]:
# Extract sample dialogues for each speaker (50-100 samples for rich context)
speaker_samples = {}

for speaker in sorted(training_data['Speaker'].unique()):
    speaker_data = training_data[training_data['Speaker'] == speaker]
    
    # Get 50-100 sample utterances with emotion/sentiment labels for rich context
    sample_size = min(max(50, len(speaker_data) // 2), min(100, len(speaker_data)))
    samples = speaker_data.sample(n=sample_size, random_state=42)[['Utterance', 'Emotion', 'Sentiment']]
    
    speaker_samples[speaker] = {
        'total_utterances': len(speaker_data),
        'sample_count': sample_size,
        'samples': samples.to_dict(orient='records'),
        'emotion_distribution': speaker_data['Emotion'].value_counts().to_dict(),
        'sentiment_distribution': speaker_data['Sentiment'].value_counts().to_dict()
    }
    
    print(f"\n{speaker}:")
    print(f"  Total utterances: {len(speaker_data)}")
    print(f"  Sampled {sample_size} for analysis")
    print(f"  Emotion distribution: {speaker_data['Emotion'].value_counts().to_dict()}")
    print(f"  Sentiment distribution: {speaker_data['Sentiment'].value_counts().to_dict()}")

In [ ]:
Role: Expert Social Psychologist & Character Profiler
Task: Analyze the provided dialogue history from the training set to construct a "Static Persona Card" for the speaker. This card will serve as the ground truth baseline for future emotion recognition.

Instructions:
Based strictly on the provided conversation turns, define the following five dimensions for the character:

Core Personality: Define their overarching temperament (e.g., neurotic, easy-going, sarcastic).

Linguistic Signature: Identify recurring speech patterns or rhetorical styles (e.g., self-deprecation, formal speech, high-energy outbursts).

Arousal Baseline: Is their "Neutral" state naturally high-energy (fast-talking/agitated) or low-energy (monotone/calm)?

Conflict Style: How do they typically express negative sentiment? (e.g., Does "Anger" manifest as direct aggression or as flustered defense/sarcasm?) .

Social Anchors: Identify 1–2 key relationships and how their tone changes when interacting with those specific people (e.g., legacy tension or protective warmth).

Input Data:
Speaker: [SPEAKER_NAME]
Dialogue Samples:
[INSERT 50-100 TURNS FROM TRAINING DATA]

Output Format (Strict JSON):

JSON
{
  "speaker": "[Name]",
  "static_persona": "[2-sentence summary of personality and role]",
  "linguistic_style": "[Key speech traits and idiosyncratic habits]",
  "baseline_arousal": "Low/Medium/High (and why)",
  "negative_expression": "[How they express anger vs. sadness vs. sarcasm]",
  "social_relationship_priors": "[Specific behavioral notes when talking to others]"
}

In [ ]:
# USER: Replace this placeholder with your actual bio card generation prompt
USER_BIO_CARD_PROMPT = """
PLACEHOLDER: This is where the user will provide their custom bio card generation prompt.

Replace this text with your actual prompt for generating bio cards.
The prompt will have access to:
  - {speaker_name}: The speaker's name
  - {total_utterances}: Total utterances for this speaker
  - {emotion_distribution}: Distribution of emotions
  - {samples}: Sample dialogues with emotion/sentiment labels
"""

print("⚠️  USER PROMPT PLACEHOLDER - Replace with your custom prompt")
print(f"Current prompt:\n{USER_BIO_CARD_PROMPT}")

In [ ]:
# Generate Bio Cards for All Speakers
bio_cards = {}
output_file = "../logs/speaker_bio_cards.txt"

# Ensure logs directory exists
os.makedirs('../logs', exist_ok=True)

# Clear/create output file
with open(output_file, 'w') as f:
    f.write("="*80 + "\n")
    f.write("SPEAKER BIO CARDS - Generated from Training Data\n")
    f.write(f"Total unique speakers: {len(speaker_samples)}\n")
    f.write("="*80 + "\n\n")

print(f"Generating bio cards for {len(speaker_samples)} speakers...\n")

for speaker_name, speaker_info in speaker_samples.items():
    print(f"Processing {speaker_name}...", end=" ")
    
    # Format samples for the prompt
    samples_text = "\n".join([
        f"  - Utterance: {s['Utterance'][:100]}... | Emotion: {s['Emotion']} | Sentiment: {s['Sentiment']}"
        for s in speaker_info['samples']
    ])
    
    # Build the full prompt
    full_prompt = BIO_CARD_PROMPT_TEMPLATE.format(
        speaker_name=speaker_name,
        total_utterances=speaker_info['total_utterances'],
        emotion_distribution=speaker_info['emotion_distribution'],
        samples=samples_text,
        user_prompt=USER_BIO_CARD_PROMPT
    )
    
    try:
        # Call LLM to generate bio card
        bio_card_text = groq_llm_call(
            prompt=full_prompt,
            model="meta-llama/llama-4-scout-17b-16e-instruct",
            api_key=BIO_CARD_API
        )
        
        bio_cards[speaker_name] = bio_card_text
        
        # Write to file
        with open(output_file, 'a') as f:
            f.write(f"\n{'='*80}\n")
            f.write(f"SPEAKER: {speaker_name}\n")
            f.write(f"Total Utterances in Training: {speaker_info['total_utterances']}\n")
            f.write(f"Emotion Distribution: {speaker_info['emotion_distribution']}\n")
            f.write(f"{'='*80}\n\n")
            f.write(bio_card_text)
            f.write("\n\n")
        
        print(f"✓ Generated")
    except Exception as e:
        print(f"✗ Error: {str(e)}")
        bio_cards[speaker_name] = f"ERROR: {str(e)}"
        with open(output_file, 'a') as f:
            f.write(f"\nERROR generating bio card for {speaker_name}: {str(e)}\n\n")

print(f"\n✓ Bio cards saved to {output_file}")

In [ ]:
# Summary of generated bio cards
print("\nBIO CARD GENERATION SUMMARY")
print("="*60)

for speaker_name, bio_card in bio_cards.items():
    status = "✓" if not bio_card.startswith("ERROR") else "✗"
    preview = bio_card[:100] if len(bio_card) > 100 else bio_card
    print(f"{status} {speaker_name}: {len(bio_card)} characters")

print(f"\n✓ All bio cards saved to: {os.path.abspath(output_file)}")

In [ ]:
# Display first bio card as sample
first_speaker = list(bio_cards.keys())[0]
print(f"\nSAMPLE BIO CARD: {first_speaker}")
print("="*80)
print(bio_cards[first_speaker][:500])
print("...\n[Full bio card saved to file]")